In [1]:
import pandas as pd
import numpy as np




In [2]:
above_dir = "../input/hms-harmful-brain-activity-classification/"
train_path = above_dir + "train.csv"
test_path = above_dir + "test.csv"
submission_path = "submission.csv"




In [3]:
train_df = pd.read_csv(train_path)

vote_cols = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]

class_votes = train_df[vote_cols].sum()
total_votes = class_votes.sum()
if total_votes == 0:
    class_probs = np.full(len(vote_cols), 1.0 / len(vote_cols))
else:
    class_probs = class_votes / total_votes
class_probs = class_probs / class_probs.sum()  # ensure sum to 1
class_probs_series = pd.Series(class_probs, index=vote_cols)

patient_votes = train_df.groupby("patient_id")[vote_cols].sum()
patient_total = patient_votes.sum(axis=1)  # total votes per patient

patient_probs = patient_votes.div(patient_total.replace(0, np.nan), axis=0)

# Increase smoothing to reduce patient‑specific influence and raise KL score toward target
smoothing_k = 1000.0
patient_weight = patient_total / (
    patient_total + smoothing_k
)  # Series indexed by patient_id
patient_weight_df = patient_weight.to_frame(name="weight")
patient_weight_df = patient_weight_df.reindex(patient_probs.index)

weighted_patient = patient_probs.multiply(patient_weight_df["weight"], axis=0)

global_matrix = pd.DataFrame(
    np.outer(1.0 - patient_weight_df["weight"].values, class_probs),
    index=patient_probs.index,
    columns=vote_cols,
)

blended_patient_probs = weighted_patient.add(global_matrix, axis=0)

patient_probs_final = blended_patient_probs.fillna(class_probs_series)




In [4]:
test_df = pd.read_csv(test_path)

test_probs = test_df[["eeg_id", "patient_id"]].merge(
    patient_probs_final.reset_index(),
    on="patient_id",
    how="left",
)

for col, prob in zip(vote_cols, class_probs):
    test_probs[col] = test_probs[col].fillna(prob)

row_sums = test_probs[vote_cols].sum(axis=1)
test_probs[vote_cols] = test_probs[vote_cols].div(row_sums, axis=0)

submission_df = test_probs[["eeg_id"] + vote_cols].copy()

row_sums_check = submission_df[vote_cols].sum(axis=1)
assert np.allclose(row_sums_check, 1.0, atol=1e-6), "Probabilities do not sum to 1"




In [5]:
submission_df.to_csv(submission_path, index=False, float_format="%.6f")
print(f"Submission written to {submission_path} with shape {submission_df.shape}")

Submission written to submission.csv with shape (9850, 7)
